<a href="https://colab.research.google.com/github/RaniellyPatricia/predicao-abandono-escolar-mg/blob/main/notebooks/02_integracao_das_bases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Integração das Bases Tratadas

## Predição do Abandono Escolar nos Municípios de Minas Gerais

Este notebook tem como objetivo integrar as bases tratadas individualmente na etapa anterior.

As bases utilizadas são:

- INSE — Nível Socioeconômico;
- TDI — Taxa de Distorção Idade-Série;
- MAT — Média de Alunos por Turma;
- RCD/IRD — Regularidade do Corpo Docente;
- Rendimento Escolar — Taxa de Abandono.

Todas as bases foram previamente tratadas para conter registros dos 853 municípios de Minas Gerais no ano de 2023.

A integração será realizada utilizando as chaves:

- `ANO`;
- `CO_MUNICIPIO`.

In [1]:
import pandas as pd

In [2]:
url_base = "https://raw.githubusercontent.com/RaniellyPatricia/predicao-abandono-escolar-mg/main/data/interim/"

url_inse = url_base + "inse_mg_2023_tratado.csv"
url_tdi = url_base + "tdi_mg_2023_tratado.csv"
url_mat = url_base + "mat_mg_2023_tratado.csv"
url_rcd = url_base + "rcd_mg_2023_tratado.csv"
url_rendimento = url_base + "rendimento_mg_2023_tratado.csv"

df_inse = pd.read_csv(url_inse)
df_tdi = pd.read_csv(url_tdi)
df_mat = pd.read_csv(url_mat)
df_rcd = pd.read_csv(url_rcd)
df_rendimento = pd.read_csv(url_rendimento)

print("INSE:", df_inse.shape)
print("TDI:", df_tdi.shape)
print("MAT:", df_mat.shape)
print("RCD:", df_rcd.shape)
print("Rendimento:", df_rendimento.shape)

INSE: (853, 5)
TDI: (853, 4)
MAT: (853, 4)
RCD: (853, 5)
Rendimento: (853, 4)


In [3]:
bases = {
    "INSE": df_inse,
    "TDI": df_tdi,
    "MAT": df_mat,
    "RCD": df_rcd,
    "Rendimento": df_rendimento
}

for nome, df in bases.items():
    print(f"\nBase: {nome}")
    print("Colunas:", list(df.columns))
    print("Anos:", df["ANO"].unique())
    print("Municípios únicos:", df["CO_MUNICIPIO"].nunique())
    print("Municípios duplicados:", df["CO_MUNICIPIO"].duplicated().sum())


Base: INSE
Colunas: ['ANO', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'QTD_ALUNOS_INSE', 'MEDIA_INSE']
Anos: [2023]
Municípios únicos: 853
Municípios duplicados: 0

Base: TDI
Colunas: ['ANO', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'TDI_ANOS_FINAIS']
Anos: [2023]
Municípios únicos: 853
Municípios duplicados: 0

Base: MAT
Colunas: ['ANO', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'MEDIA_ALUNOS_TURMA_ANOS_FINAIS']
Anos: [2023]
Municípios únicos: 853
Municípios duplicados: 0

Base: RCD
Colunas: ['ANO', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'PERC_DOCENTES_BAIXA_REGULARIDADE', 'PERC_DOCENTES_ALTA_REGULARIDADE']
Anos: [2023]
Municípios únicos: 853
Municípios duplicados: 0

Base: Rendimento
Colunas: ['ANO', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'TAXA_ABANDONO_ANOS_FINAIS']
Anos: [2023]
Municípios únicos: 853
Municípios duplicados: 0


In [4]:
municipios_inse = set(df_inse["CO_MUNICIPIO"])
municipios_tdi = set(df_tdi["CO_MUNICIPIO"])
municipios_mat = set(df_mat["CO_MUNICIPIO"])
municipios_rcd = set(df_rcd["CO_MUNICIPIO"])
municipios_rendimento = set(df_rendimento["CO_MUNICIPIO"])

print("INSE x TDI:", municipios_inse == municipios_tdi)
print("INSE x MAT:", municipios_inse == municipios_mat)
print("INSE x RCD:", municipios_inse == municipios_rcd)
print("INSE x Rendimento:", municipios_inse == municipios_rendimento)

INSE x TDI: True
INSE x MAT: True
INSE x RCD: True
INSE x Rendimento: True


In [5]:
df_dataset = df_inse.merge(
    df_tdi[["ANO", "CO_MUNICIPIO", "TDI_ANOS_FINAIS"]],
    on=["ANO", "CO_MUNICIPIO"],
    how="inner",
    validate="one_to_one"
)

df_dataset = df_dataset.merge(
    df_mat[["ANO", "CO_MUNICIPIO", "MEDIA_ALUNOS_TURMA_ANOS_FINAIS"]],
    on=["ANO", "CO_MUNICIPIO"],
    how="inner",
    validate="one_to_one"
)

df_dataset = df_dataset.merge(
    df_rcd[
        [
            "ANO",
            "CO_MUNICIPIO",
            "PERC_DOCENTES_BAIXA_REGULARIDADE",
            "PERC_DOCENTES_ALTA_REGULARIDADE"
        ]
    ],
    on=["ANO", "CO_MUNICIPIO"],
    how="inner",
    validate="one_to_one"
)

df_dataset = df_dataset.merge(
    df_rendimento[["ANO", "CO_MUNICIPIO", "TAXA_ABANDONO_ANOS_FINAIS"]],
    on=["ANO", "CO_MUNICIPIO"],
    how="inner",
    validate="one_to_one"
)

print("Formato do dataset final:", df_dataset.shape)
print("Municípios únicos:", df_dataset["CO_MUNICIPIO"].nunique())
print("Municípios duplicados:", df_dataset["CO_MUNICIPIO"].duplicated().sum())

df_dataset.head()

Formato do dataset final: (853, 10)
Municípios únicos: 853
Municípios duplicados: 0


,ANO,CO_MUNICIPIO,NO_MUNICIPIO,QTD_ALUNOS_INSE,MEDIA_INSE,TDI_ANOS_FINAIS,MEDIA_ALUNOS_TURMA_ANOS_FINAIS,PERC_DOCENTES_BAIXA_REGULARIDADE,PERC_DOCENTES_ALTA_REGULARIDADE,TAXA_ABANDONO_ANOS_FINAIS
0,2023,3136959,Juvenília,228,4.4254,8.8,17.5,0.0,27.3,0.8
1,2023,3136900,Juruaia,306,5.1850,17.4,31.6,14.3,0.0,3.4
2,2023,3136702,Juiz de Fora,9283,5.1931,19.2,24.9,11.1,2.8,1.2
3,2023,3136652,Juatuba,966,5.1363,8.0,24.2,23.5,11.8,0.1
4,2023,3136603,Nova União,191,5.0039,22.5,28.8,0.0,0.0,0.6


In [6]:
print("Valores ausentes por coluna:")
print(df_dataset.isna().sum())

print("\nTipos das colunas:")
print(df_dataset.dtypes)

print("\nAnos encontrados:")
print(df_dataset["ANO"].unique())

Valores ausentes por coluna:
ANO                                 0
CO_MUNICIPIO                        0
NO_MUNICIPIO                        0
QTD_ALUNOS_INSE                     0
MEDIA_INSE                          0
TDI_ANOS_FINAIS                     0
MEDIA_ALUNOS_TURMA_ANOS_FINAIS      0
PERC_DOCENTES_BAIXA_REGULARIDADE    0
PERC_DOCENTES_ALTA_REGULARIDADE     0
TAXA_ABANDONO_ANOS_FINAIS           0
dtype: int64

Tipos das colunas:
ANO                                   int64
CO_MUNICIPIO                          int64
NO_MUNICIPIO                         object
QTD_ALUNOS_INSE                       int64
MEDIA_INSE                          float64
TDI_ANOS_FINAIS                     float64
MEDIA_ALUNOS_TURMA_ANOS_FINAIS      float64
PERC_DOCENTES_BAIXA_REGULARIDADE    float64
PERC_DOCENTES_ALTA_REGULARIDADE     float64
TAXA_ABANDONO_ANOS_FINAIS           float64
dtype: object

Anos encontrados:
[2023]


In [7]:
df_dataset[
    df_dataset["NO_MUNICIPIO"].str.upper() == "FRUTAL"
]

,ANO,CO_MUNICIPIO,NO_MUNICIPIO,QTD_ALUNOS_INSE,MEDIA_INSE,TDI_ANOS_FINAIS,MEDIA_ALUNOS_TURMA_ANOS_FINAIS,PERC_DOCENTES_BAIXA_REGULARIDADE,PERC_DOCENTES_ALTA_REGULARIDADE,TAXA_ABANDONO_ANOS_FINAIS
98,2023,3127107,Frutal,1475,5.1652,13.4,26.2,0.0,4.3,1.5


In [8]:
df_dataset.to_csv(
    "dataset_abandono_mg_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dataset final salvo com sucesso.")

Dataset final salvo com sucesso.


In [9]:
from google.colab import files

files.download("dataset_abandono_mg_2023.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>